In [ ]:
!pip install torch torchvision datasets transformers scikit-learn matplotlib


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, accuracy_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [ ]:
# Custom Dataset class for Hugging Face dataset
class SST2Dataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.texts = dataset['sentence']
        self.labels = dataset['label']  # For accuracy calculation
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': self.labels[idx]
        }


In [ ]:
# Autoencoder Model
class Autoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, latent_size):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, latent_size),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
            nn.Tanh()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent


In [ ]:
# Set hyperparameters
input_size = 768  # BERT embedding size
hidden_size = 256
latent_size = 64
learning_rate = 0.001
batch_size = 32
epochs = 5
num_clusters = 2  # Binary sentiment clustering
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load dataset and tokenizer
dataset = load_dataset("gimmaru/glue-sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)



In [ ]:
# Dynamically select the first available split
available_splits = list(dataset.keys())
if not available_splits:
    raise ValueError("No splits available in the dataset")
data_split = available_splits[0]  # Use the first split (e.g., 'train', 'data', etc.)
print(f"Using dataset split: {data_split}")

Using dataset split: validation


In [ ]:
# Create dataloaders
train_dataset = SST2Dataset(dataset[data_split], tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Initialize model, loss function, and optimizer
model = Autoencoder(input_size, hidden_size, latent_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
# Training loop
def train_autoencoder():
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            with torch.no_grad():
                outputs = bert_model(input_ids, attention_mask=attention_mask)
                embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token

            reconstructed, _ = model(embeddings)
            loss = criterion(reconstructed, embeddings)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Get latent embeddings and labels for clustering
def get_latent_embeddings_and_labels():
    model.eval()
    embeddings = []
    true_labels = []
    with torch.no_grad():
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']
            outputs = bert_model(input_ids, attention_mask=attention_mask)
            bert_embeddings = outputs.last_hidden_state[:, 0, :]
            _, latent = model(bert_embeddings)
            embeddings.append(latent.cpu().numpy())
            true_labels.extend(labels.numpy() if hasattr(labels, 'numpy') else labels)
    return np.vstack(embeddings), np.array(true_labels)

In [ ]:
# Training and clustering
print("Training autoencoder...")
train_autoencoder()

print("Extracting latent embeddings...")
latent_embeddings, true_labels = get_latent_embeddings_and_labels()


Training autoencoder...
Epoch [1/5], Loss: 0.1777
Epoch [2/5], Loss: 0.1389
Epoch [3/5], Loss: 0.1350
Epoch [4/5], Loss: 0.1315
Epoch [5/5], Loss: 0.1289
Extracting latent embeddings...


In [ ]:
# Perform K-Means clustering
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(latent_embeddings)

In [ ]:

# Map clusters to true labels for accuracy
def map_clusters_to_labels(cluster_labels, true_labels, num_clusters):
    from scipy.stats import mode
    labels = np.zeros_like(cluster_labels)
    for i in range(num_clusters):
        mask = (cluster_labels == i)
        labels[mask] = mode(true_labels[mask])[0]
    return labels

mapped_labels = map_clusters_to_labels(cluster_labels, true_labels, num_clusters)
clustering_accuracy = accuracy_score(true_labels, mapped_labels)

In [ ]:
# Evaluation metrics
silhouette = silhouette_score(latent_embeddings, cluster_labels)
db_index = davies_bouldin_score(latent_embeddings, cluster_labels)

print(f"Clustering Accuracy: {clustering_accuracy:.4f}")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"Davies-Bouldin Index: {db_index:.4f}")

Clustering Accuracy: 0.6330
Silhouette Score: 0.1652
Davies-Bouldin Index: 2.0346


In [ ]:
# Visualize clusters using t-SNE
tsne = TSNE(n_components=2, random_state=42)
tsne_embeddings = tsne.fit_transform(latent_embeddings)

plt.figure(figsize=(10, 8))
plt.scatter(tsne_embeddings[:, 0], tsne_embeddings[:, 1], c=cluster_labels, cmap='viridis')
plt.title('t-SNE Visualization of Clusters')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar()
plt.savefig('cluster_visualization.png')
plt.close()

print("Cluster visualization saved as 'cluster_visualization.png'")

Cluster visualization saved as 'cluster_visualization.png'
